# Stochastic Quantization VAE（SQ-VAE）

---
## 目的
`vq_vae.ipynb`のVQ-VAEが抱える課題（Straight-Through Estimatorによる勾配の近似や，コミットメント損失の重みを人手で調整する必要がある点）を，確率的な量子化と自己アニーリングする分散パラメータによって解消するSQ-VAE [1]を構築する．`vq_vae.ipynb`と全く同じネットワーク構造を用いることで，両者の違いを比較する．

[1] Yuhta Takida, Takashi Shibuya, Wei-Hsiang Liao, Chieh-Hsin Lai, Junki Ohmura, Toshimitsu Uesaka, Naoki Murata, Shusuke Takahashi, Toshiyuki Kumakura, Yuki Mitsufuji, "SQ-VAE: Variational Bayes on Discrete Representation with Self-annealed Stochastic Quantization," ICML, 2022.\
[2] Aaron van den Oord, Oriol Vinyals, Koray Kavukcuoglu, "Neural Discrete Representation Learning," NeurIPS, 2017.

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
import time
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## SQ-VAEとは
`vq_vae.ipynb`のVQ-VAEは，Encoderの出力を最も近いコードブックのベクトルへ**決定的に**（argminにより一意に）置き換えていました．この量子化は微分できないため，Straight-Through Estimator（勾配をそのままコピーする近似）が必要でした．また，コードブックを更新する「コードブック損失」とEncoderを更新する「コミットメント損失」のバランスは，`commitment_cost`という人手で調整するハイパーパラメータに依存していました．

SQ-VAE (Stochastic Quantization VAE) [1] は，量子化を「決定的な最近傍選択」ではなく「**確率的な選択**」として定式化することで，これらの課題に対処します．具体的には，各コードブックのベクトル$e_k$を選択する確率を，以下のようなカテゴリカル分布として表します．

$$
q(k \mid x) = \frac{\exp\left(-\|z_e - e_k\|^2 / 2\sigma^2\right)}{\sum_{k'} \exp\left(-\|z_e - e_{k'}\|^2 / 2\sigma^2\right)}
$$

ここで$\sigma^2$は学習可能なパラメータです．$\sigma^2$が大きいときはほぼ一様な（曖昧な）確率分布となり，$\sigma^2 \to 0$に近づくほど，最も距離が近いコードのみが選ばれる決定的な分布（＝VQ-VAEのargminと同じ）に近づきます．

## 自己アニーリングする分散パラメータ
SQ-VAEでは，$\sigma^2$を人手で決めるのではなく，以下の損失項を通じて学習可能なパラメータとして最適化します（$D$はベクトルの次元数）．

$$
\mathcal{L}_{\sigma} = \frac{1}{2\sigma^2}\left(\text{コードブック損失} + \text{コミットメント損失}\right) + \frac{D}{2}\log \sigma^2
$$

この式は，VQ-VAEの損失$\mathcal{L}_{vq} = \text{コードブック損失} + \beta\cdot\text{コミットメント損失}$と似た形をしていますが，重み$\beta$の代わりに$\frac{1}{2\sigma^2}$が使われており，さらに$\frac{D}{2}\log\sigma^2$という項が追加されています．この項があることで，コードブックとEncoderの出力の誤差（コードブック損失＋コミットメント損失）が小さくなるにつれて，$\sigma^2$を小さくする（＝量子化をより決定的にする）ことが損失全体を下げるようになります．つまり，学習が進むにつれて$\sigma^2$が自動的に小さくなっていきます．これを**自己アニーリング（self-annealing）**と呼びます．学習の初期は確率的な（曖昧な）量子化により多くのコードが試され，学習が進むにつれて決定的な量子化（VQ-VAEに近い挙動）へと自然に移行していきます．

実際に確率分布からサンプリングする際は，カテゴリカル分布からのサンプリングをそのままでは誤差逆伝播できないため，**Gumbel-Softmax**という手法（カテゴリカル分布に対するReparameterization trickに相当する手法）を用いて，微分可能な形でサンプリングします．

In [ ]:
class StochasticQuantizer(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1.0 / num_embeddings, 1.0 / num_embeddings)
        self.log_var = nn.Parameter(torch.zeros(1))  # 自己アニーリングする分散（の対数）

    def forward(self, z_e):
        b, d, h, w = z_e.shape
        flat_z_e = z_e.permute(0, 2, 3, 1).reshape(-1, d)

        distances = (flat_z_e.pow(2).sum(1, keepdim=True)
                     - 2 * flat_z_e @ self.embedding.weight.t()
                     + self.embedding.weight.pow(2).sum(1))
        var = self.log_var.exp()
        logits = -distances / (2 * var)

        if self.training:
            # Gumbel-Softmaxを用いて，確率的かつ微分可能にコードを選択する
            weights = F.gumbel_softmax(logits, tau=1.0, hard=True)
        else:
            # 評価時は決定的に最も確率の高い（=最も距離が近い）コードを選択する
            eval_indices = torch.argmin(distances, dim=1)
            weights = F.one_hot(eval_indices, self.num_embeddings).float()

        z_q = (weights @ self.embedding.weight).view(b, h, w, d).permute(0, 3, 1, 2)

        codebook_loss = F.mse_loss(z_q, z_e.detach())
        commitment_loss = F.mse_loss(z_e, z_q.detach())
        quant_loss = (codebook_loss + commitment_loss) / (2 * var.squeeze()) + 0.5 * d * self.log_var.squeeze()

        # Straight-Through Estimator（Decoderへの勾配をEncoderの出力へそのまま橋渡しする）
        z_q = z_e + (z_q - z_e).detach()

        # コードブックの利用状況（Perplexity）を計算
        probs = F.softmax(logits, dim=1)
        avg_probs = probs.mean(0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))

        indices = torch.argmin(distances, dim=1)
        return z_q, quant_loss, perplexity, var, indices.view(b, h, w)

## ネットワークの作成
Encoder・Decoderの構造は，`vq_vae.ipynb`のVQ-VAEと全く同じです（畳み込み4層，`embedding_dim=64`, `num_embeddings=128`）．量子化を行う部分のみを，`VectorQuantizer`から`StochasticQuantizer`に置き換えます．

In [ ]:
class SQVAE(nn.Module):
    def __init__(self, embedding_dim=64, num_embeddings=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 32 -> 16
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 16 -> 8
            nn.Conv2d(64, embedding_dim, kernel_size=3, stride=1, padding=1),  # 8 -> 8
        )
        self.sq = StochasticQuantizer(num_embeddings, embedding_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 8 -> 16
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True),  # 16 -> 32
            nn.Conv2d(32, 1, kernel_size=3, stride=1, padding=1),  # Sigmoidは適用しない（誤差関数側でまとめて適用する）
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, quant_loss, perplexity, var, indices = self.sq(z_e)
        x_hat = self.decoder(z_q)
        return x_hat, quant_loss, perplexity, var, indices

## データセット，ネットワーク，最適化関数の設定
`vq_vae.ipynb`と同じ設定（MNISTを$32\times32$にリサイズ，Adam optimizer，学習率$2\times 10^{-4}$）を使用します．

In [ ]:
batch_size = 128

transform = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])
mnist_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(mnist_data, batch_size=batch_size, shuffle=True)

model = SQVAE(embedding_dim=64, num_embeddings=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

## 学習
誤差関数は，再構成誤差と，`sq`モジュールが返す`quant_loss`（自己アニーリングする分散$\sigma^2$で重み付けされた量子化誤差）の和です．学習の進行とともに`var`（$\sigma^2$）がどのように変化していくかにも注目してください．学習エポック数を`30`とします．

In [ ]:
epoch_num = 30

model.train()
start = time.time()
for epoch in range(1, epoch_num + 1):
    sum_loss, sum_recon, sum_quant, sum_ppl = 0.0, 0.0, 0.0, 0.0
    for x, _ in train_loader:
        x = x.to(device)

        x_hat, quant_loss, perplexity, var, _ = model(x)
        recon_loss = F.binary_cross_entropy_with_logits(x_hat, x, reduction='mean')
        loss = recon_loss + quant_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_recon += recon_loss.item()
        sum_quant += quant_loss.item()
        sum_ppl += perplexity.item()

    n = len(train_loader)
    print(f'epoch: {epoch}, loss: {sum_loss/n:.4f}, recon: {sum_recon/n:.4f}, quant_loss: {sum_quant/n:.4f}, perplexity: {sum_ppl/n:.2f}, var: {var.item():.5f}, elapsed_time: {time.time()-start:.2f}')

## 学習済みモデルを用いた画像の復元
`vq_vae.ipynb`と同様に，評価用データからランダムに画像をサンプルし，再構成結果を確認します．

In [ ]:
mnist_testdata = datasets.MNIST(root='./data', train=False, transform=transform)
test_loader = DataLoader(mnist_testdata, batch_size=10, shuffle=True)

model.eval()
with torch.no_grad():
    x, _ = next(iter(test_loader))
    x = x.to(device)
    x_hat, quant_loss, perplexity, var, indices = model(x)
    x_hat = torch.sigmoid(x_hat)

x_cpu = x.cpu()
x_hat_cpu = x_hat.cpu()

fig, axes = plt.subplots(2, 10, figsize=(14, 2.8))
for i in range(10):
    axes[0, i].imshow(x_cpu[i, 0], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(x_hat_cpu[i, 0], cmap='gray'); axes[1, i].axis('off')
fig.suptitle('input (top) / reconstruction (bottom)')
plt.show()

## 離散潜在表現（コード割り当て）の可視化
`vq_vae.ipynb`と同様に，評価時（決定的な量子化）で選択されたコード番号の並びを可視化します．

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for i in range(10):
    axes[0, i].imshow(x_cpu[i, 0], cmap='gray'); axes[0, i].axis('off')
    axes[1, i].imshow(indices[i].cpu(), cmap='tab20'); axes[1, i].axis('off')
fig.suptitle('input (top) / assigned code indices (bottom)')
plt.show()

## 課題

1. 学習の各エポックで$\sigma^2$（`var`）の値を記録し，学習の進行とともにどのように変化するかグラフにプロットして確認してください．
2. `vq_vae.ipynb`のVQ-VAEと本ノートブックのSQ-VAEで，最終的なPerplexity（コードブックの利用状況）を比較してください．
3. `log_var`の初期値を変更して学習し，学習の安定性や収束の速さにどのような影響があるか確認してください．